# Create finetuned models

In [3]:
# Import basic libraries
import numpy as np
import pandas as pd
import scipy as sc
from itertools import product
import os
from sentence_transformers import SentenceTransformer
import time
import torch
import sys
import random
import torch
from enum import Enum
from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments
)
from sentence_transformers.losses import ContrastiveLoss, TripletLoss, AnglELoss

We finetune models, to enhance their ability to create meaningful distinctions in our dataset. To motivate this, let us begin with looking at a 2D representation of where jina-embeddings-v2-small-en places the survey responses in the embedding space.

In [ ]:
# initialize model and prompt
# Note: The model_name and prompt can be changed as needed.

model_name = 'jinaai/jina-embeddings-v2-small-en'
prompt = ''

model = SentenceTransformer(
    model_name,
    trust_remote_code=True
)

In [ ]:
#Tsne before finetuning
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from sklearn.manifold import TSNE

# ---------------------------------------------------------------------------
rcParams.update({
    "figure.dpi": 600,
    "savefig.dpi": 600,
    "font.size": 8,              # body text size
    "axes.labelsize": 6,
    "axes.titlesize": 8,
    "legend.fontsize": 6,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.5,
    "grid.linewidth": 0.5,
    "lines.markersize": 3.0,
})


# ---------------------------------------------------------------------------
# Internal helpers

_MARKERS = [
    "o", "s", "D", "X", "^", "v", ">", "<", "P", "*", "+", "x"
]


def _style_lookup(codes, palette=None, markers=None):
    """Map each unique code to a colour and a marker symbol."""
    unique = list(dict.fromkeys(codes))  # preserve original order

    # --- colours ---
    if palette is None:
        palette = sns.color_palette("tab10", len(unique))
    elif isinstance(palette, dict):
        palette = [palette.get(c, "#cccccc") for c in unique]
    colours = dict(zip(unique, palette))

    # --- markers ---
    if isinstance(markers, dict):
        marker_map = {c: markers.get(c, "o") for c in unique}
    else:
        marker_cycle = markers if markers is not None else _MARKERS
        marker_map = {
            c: marker_cycle[i % len(marker_cycle)] for i, c in enumerate(unique)
        }
    return colours, marker_map


def _scatter_plot(
    xy,
    codes,
    *,
    width_cm=8.6,
    palette=None,
    markers=None,
    x_label="Component 1",
    y_label="Component 2",
    title=None,
    save_as=None,
    show_grid=True,
):
    """Internal plotting engine; returns (fig, ax)."""
    colours, marker_map = _style_lookup(codes, palette, markers)

    # Seaborn styling
    style = "whitegrid" if show_grid else "white"
    sns.set_theme()

    # Figure size
    width_in = width_cm / 2
    height_in = width_in  # square by default – override as needed

    fig, ax = plt.subplots(figsize=(width_in, width_in))

    for code in np.unique(codes):
        mask = np.asarray(codes) == code
        ax.scatter(
            xy[mask, 0],
            xy[mask, 1],
            c=[colours[code]],
            marker=marker_map[code],
            s=18,  # ≈ 3 pt at 600 dpi
            edgecolors="none",
            label=code,
        )

    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    
    # --- hide tick labels & marks ---
    ax.set_xticks([])      # or set_xticklabels([])
    ax.set_yticks([])      # or set_yticklabels([])

    # keep everything else the same…
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)
    ax.tick_params(width=0.5)

    if title:
        ax.set_title(title, pad=4)
        
    # Square aspect so data units are equal on both axes
    ax.set_aspect("equal", adjustable="box")

    # Clean up spines but keep axes visible
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)
    ax.tick_params(width=0.5)

    ax.legend(
        title="Code",
        frameon=False,
        handletextpad=0.3,
        labelspacing=0.4,
        borderpad=0.2,
        loc="upper left",
        bbox_to_anchor=(1.02, 1),  # right of axes
        borderaxespad=0.0,
    )


    plt.tight_layout(pad=0.2)

    if save_as:
        fig.savefig(save_as, bbox_inches="tight")
        
    return fig, ax


# ---------------------------------------------------------------------------
# Public API functions
# ---------------------------------------------------------------------------

def plot_tsne_embeddings(
    embedding_matrix,
    codes,
    *,
    random_state=0,
    perplexity=30,
    width_cm=8.6,
    palette=None,
    markers=None,
    save_as=None,
):
    """t‑SNE projection formatted for a PNAS column.

    Parameters
    ----------
    embedding_matrix : (n_samples, n_features) array‑like
    codes            : iterable of length n_samples
    random_state     : int, default 0
    perplexity       : float, default 30
    width_cm         : figure width in centimetres (8.6 cm = single column)
    palette, markers : dict / list / None (see _style_lookup)
    save_as          : str or path; if given, saves figure
    """
    tsne = TSNE(
        n_components=2,
        random_state=random_state,
        perplexity=perplexity,
        init="pca",
    )
    xy = tsne.fit_transform(embedding_matrix)

    return _scatter_plot(
        xy,
        codes,
        width_cm=width_cm,
        palette=palette,
        markers=markers,
        x_label="t‑SNE 1",
        y_label="t‑SNE 2",
        title=None,  # no title; let LaTeX caption handle it
        save_as=save_as,
        show_grid=True,
    )

## Step 1. Create a finetune dataset 

In [ ]:
def create_finetune_dataset():

    # Import the data
    df = pd.read_excel('data/centroids.xlsx')

    # create a list of all the texts
    texts = df['text'].tolist()
    combinations = list(product(texts, repeat=2))

    # create a df with the combinations
    combinations_df = pd.DataFrame(combinations, columns=['anchor', 'target'])

    # create a function that checks if the anchor and target have the same code in centroids_df
    def check_code(anchor, target):
        anchor_code = df[df['text'] == anchor]['code'].values[0]
        target_code = df[df['text'] == target]['code'].values[0]
        if anchor_code == target_code:
            return float(1)
        else:
            return float(0)
        
    # apply the function to the df
    combinations_df['label'] = combinations_df.apply(lambda x: check_code(x['anchor'], x['target']), axis=1)

    combinations_df.rename(columns={'anchor': 'text1','target': 'text2'}, inplace=True)

    # save the df to a csv
    combinations_df.to_csv('data/finetune_dataset.csv', index=False)



In [ ]:
Args = {
    "model": model,
    "prompt": prompt,
}

## Create ft dataset

In [ ]:
# import centroids + additional texts


## Do finetuning

In [ ]:
# Set args

# Set loss

# Set output dir

## Tsne